# Pre-Trained LLM Tuning and Evaluation for Question Answering
- **Student:** `Hunter Worssam`  
- **Date:** `2025-11-16`  

## README 

- **Python version:** `3.10.18`
- **Standard library modules:**  
  `sys`, `platform`, `warnings`, `csv`, `re`, `os`, `logging`, `difflib`
- **Third-party packages:**
  - `numpy`
  - `pandas`
  - `matplotlib`
  - `datasets` (Hugging Face Datasets)
  - `torch` (PyTorch)
  - `transformers` (Hugging Face Transformers)
  - `accelerate` (used under the hood by `Trainer`)

## Install instructions

- **pip install numpy pandas matplotlib datasets torch transformers accelerate**

- **Datasets used:**
  - Load the dataset using the Hugging Face datasets library rather than downloading a local file.
  - https://huggingface.co/datasets/dkasinets/alice_in_wonderland_qa
    
- **How to run this notebook:**
  1. Run all cells in order (Kernel → Restart & Run All).
  2. Ensure figures and tables render correctly.
```bash

In [4]:
# Imports
import sys, platform
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import csv
import re
import os
from datasets import load_dataset
from difflib import SequenceMatcher
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    import numpy as np, pandas as pd
    print("NumPy:", np.__version__)
    print("Pandas:", pd.__version__)
except Exception as e:
    print("Optional packages missing or version check failed:", e)

# PyTprch
import torch
print(f'torch version = {torch.__version__}')
print(f'cuda available = {torch.cuda.is_available()}')
import transformers
from transformers import pipeline
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import EarlyStoppingCallback

# filter warnings
from warnings import simplefilter
simplefilter(action='ignore', category=UserWarning)
print(f'transformers version = {transformers.__version__}')
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings(
    "ignore", 
    message=".*IProgress not found.*"
)

/opt/anaconda3/envs/research_disc_one_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.10.18
Platform: macOS-15.6.1-arm64-arm-64bit
NumPy: 1.26.4
Pandas: 2.3.2
torch version = 2.3.1
cuda available = False
transformers version = 4.43.3


In [5]:
!pip install transformers accelerate -q

In [6]:
!pip install accelerate -U

In [7]:
!pip install "transformers[torch]"

# __1. Load the Dataset__<br>
Get the *Alice in Wonderland* dataset from the [Alice in Wonderland QA dataset](https://huggingface.co/datasets/dkasinets/alice_in_wonderland_qa), which contains questions and answers.

You can access it using the Hugging Face `datasets` library.

This code below loads the Alice in Wonderland question-answer dataset using the Hugging Face datasets library and then filters the entries to keep only those whose source column contains the word “Alice.” Because the dataset’s available fields are question, context, answer, and source, the filtering step checks the source column, where chapter titles and book names are stored, to identify QA pairs specifically associated with Alice’s Adventures in Wonderland. The result is a subset of the original dataset that includes only the relevant questions, contexts, and answers tied to chapters whose titles reference “Alice.”

In [10]:
# Load the dataset
dataset = load_dataset("dkasinets/alice_in_wonderland_qa")

# Take the "train" split within
data = dataset["train"]

print(f' Column names: {data.column_names}')

# Filter rows where source contains "Alice" 
filtered = data.filter(lambda x: "alice" in x["source"].lower())

# Check how many matched
print(f' Total # of entries: {len(data)}')
print(f' Total # of entries featuring "alice" in the source column: {len(filtered)}')

# Peek at first few entries
print(f'\n Examples:')
filtered[:3]

 Column names: ['question', 'context', 'answer', 'source']
 Total # of entries: 129
 Total # of entries featuring "alice" in the source column: 123

 Examples:


{'question': ["Which character sings 'Twas Brillig'?",
  "What condiment did the Mad Hatter think was ridiculous to put inside the White Rabbit's watch?",
  "What is Alice's cat's name?"],
 'context': ["The poem 'Jabberwocky' begins with the line ''Twas brillig'. It appears in Through the Looking-Glass, and while no one sings it aloud, the poem is presented to Alice. It is not specifically sung by the Cheshire Cat.",
  "At the tea party, the Hatter inspects the White Rabbit's watch and says it's full of butter and other things like crumbs and mustard. He declares that mustard is the most ridiculous thing in it.",
  'Alice often talks about her cat Dinah in the book. She tells the animals in Wonderland how good Dinah is at catching birds and mice.'],
 'answer': ["No character sings it, but the poem 'Jabberwocky' appears in Through the Looking-Glass.",
  'Mustard',
  'Dinah'],
 'source': ['https://www.funtrivia.com/en/Movies/Alice-in-Wonderland-7156.html',
  'https://www.funtrivia.com/en

Of 129 total questions, our filtered dataset includes 123 entries that feature the word "Alice" in the source column. 

# __2. Load Google's FLAN-T5-small model from HuggingFace__<br>

To work with a lightweight language model suitable for tasks like question answering, summarization, or sentiment analysis, I loaded the FLAN-T5-Small model from Hugging Face. FLAN-T5 is an instruction-tuned transformer model that accepts natural-language prompts and generates text outputs, making it flexible for many downstream NLP tasks. The code below initializes the model and tokenizer using the transformers library and sets up a text2text-generation pipeline, which provides a simple interface for sending prompts to the model and receiving generated responses. This setup enables efficient experimentation on a standard notebook environment while keeping compute requirements low since I do not have any GPU's setup for use with this notebook.

In [14]:
model_name = "google/flan-t5-small"

# Load tokenizer and model from Hugging Face
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Create a text2text-generation pipeline
text2text = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
)

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


# __3. Trivia Questions__<br>
Let's first test the model on trivia questions unrelated to the novel to check general performance.

Baseline Questions:
- "Which American city is the Statue of Liberty located in?"
- "What is the capital of France?"
- "Who is the president of the United States?"
- "What is the most probable word coming after 'Alice in'?"

This code below evaluates the model’s baseline performance by prompting it with a set of general trivia questions unrelated to the Alice in Wonderland dataset. Each question is converted into a simple instruction-style prompt and passed through the text2text generation pipeline, which produces the model’s predicted answer. The loop prints each question alongside the model’s generated response, allowing us to quickly assess how well the pre-trained FLAN-T5 model handles basic queries before applying it to Alice in Wonderland specific tasks.

In [17]:
trivia_questions = [
    "Which American city is the Statue of Liberty located in?",
    "What is the capital of France?",
    "Who is the president of the United States?",
    "What is the most probable word coming after 'Alice in'?"
]

for q in trivia_questions:
    prompt = f"Question: {q}\nAnswer:"
    result = text2text(prompt, max_new_tokens=40)
    answer = result[0]["generated_text"]
    print(f"Q: {q}\nA: {answer}\n{'-'*60}")

Q: Which American city is the Statue of Liberty located in?
A: san diego
------------------------------------------------------------
Q: What is the capital of France?
A: london
------------------------------------------------------------
Q: Who is the president of the United States?
A: john d. bush
------------------------------------------------------------
Q: What is the most probable word coming after 'Alice in'?
A: apocalypse
------------------------------------------------------------


When tested on simple trivia questions, the small FLAN-T5 model produced coherent but factually incorrect answers (e.g., ‘san diego’ for the location of the Statue of Liberty and ‘london’ as the capital of France). This highlights the limitations of small, stand-alone LLMs for factual QA and illustrates how they rely on learned text patterns rather than explicit knowledge retrieval. There is plenty of room left for fine tuning to boost performance, starting with things like removing or reducing input sampling. The general structure of the answers were all in line with expectations, they were simply factually incorrect; something you can expect from a lightweight 80-million parameter model like FLAN-T5-small.  

# __4. Novel-Based Questions (Before Fine-Tuning)__<br>
Now, test the model's performance on questions about Alice in Wonderland, before fine-tuning.

Novel Questions:
- "What is the White Rabbit's catchphrase?"
- "Why does Alice shrink?"
- "What is the Queen's full name?"
- "What game does Alice play?"
- "Who wrote Alice's Adventures in Wonderland?"
- "Who played the Mad Hatter in Tim Burton's film version of Alice's Adventures in Wonderland?"

This code evaluates the pre-trained FLAN-T5-Small model on a set of questions specifically about Alice’s Adventures in Wonderland and related media using the same structure as in part 3. Each prompt is passed to the text2text pipeline, which generates an answer using the model, and the loop prints both the original question and the model’s response, separated by a line of dashes for readability.

In [21]:
novel_questions = [
    "What is the White Rabbit's catchphrase?",
    "Why does Alice shrink?",
    "What is the Queen's full name?",
    "What game does Alice play?",
    "Who wrote Alice's Adventures in Wonderland?",
    "Who played the Mad Hatter in Tim Burton's film version of Alice's Adventures in Wonderland?"
]

for q in novel_questions:
    prompt = f"Question: {q}\nAnswer:"
    result = text2text(prompt, max_new_tokens=40)
    answer = result[0]["generated_text"]
    print(f"Q: {q}\nA: {answer}\n{'-'*60}")

Q: What is the White Rabbit's catchphrase?
A: he is the king
------------------------------------------------------------
Q: Why does Alice shrink?
A: a swollen swollen swollen swollen swollen swollen swollen swollen swollen s
------------------------------------------------------------
Q: What is the Queen's full name?
A: elizabeth ii
------------------------------------------------------------
Q: What game does Alice play?
A: sand
------------------------------------------------------------
Q: Who wrote Alice's Adventures in Wonderland?
A: john scott
------------------------------------------------------------
Q: Who played the Mad Hatter in Tim Burton's film version of Alice's Adventures in Wonderland?
A: john scott
------------------------------------------------------------


The resulting outputs are clearly poor and mostly incorrect (e.g., “a swollen swollen…” for why Alice shrinks, “john scott” as both the author and actor, “sand” as the game), which highlights the limitations of a small stand-alone model used without any task- or domain-specific adaptation. FLAN-T5-Small has limited capacity and no access to external knowledge or the actual text of the novel, so it relies on language patterns and tends to hallucinate plausible-sounding but wrong answers. In principle, performance could be improved by fine-tuning the model on the Alice QA pairs from the filtered dataset, by giving it relevant context passages alongside each question (retrieval-augmented QA), by using a larger model with more parameters, or by refining prompts and decoding settings (e.g., greedy decoding, stricter answer formats) to reduce randomness and repetition.

# __5. Fine-Tuning the Model on the Alice in Wonderland__<br>
Next we use the HuggingFace's `Trainer` and `Dataset` APIs to fine-tune the model on *Alice in Wonderland*-related questions from the <a href="https://huggingface.co/datasets/dkasinets/alice_in_wonderland_qa" target="_blank">Alice in Wonderland QA dataset</a> .

To improve the model’s performance on Alice in Wonderland–specific questions, my primary approach is to fine-tune FLAN-T5-Small using the Hugging Face Trainer API and the curated Alice QA dataset. Fine-tuning allows the model to directly learn domain-specific question–answer patterns rather than relying solely on its limited pre-trained knowledge. After evaluating the fine-tuned model’s results, I will layer in additional techniques if the answer quality still needs to be improved. These additional techniques include enhanced prompt engineering and retrieval-augmented QA. This staged strategy mirrors a realistic workflow: begin with supervised adaptation, then incrementally incorporate complementary methods to address any remaining performance gaps.

### Fine Tuning via HuggingFace Trainer

The section below prepares the FLAN-T5-Small model and the Alice in Wonderland dataset for fine-tuning. It first loads the pretrained tokenizer and sequence-to-sequence model, then splits the filtered dataset into an 80/20 training and validation set. The preprocess_batch function converts each example into the input–output format required for supervised training: it concatenates each question with its corresponding context passage, tokenizes the combined input text, and separately tokenizes the answer as the target sequence. These tokenized inputs and labels are truncated to fixed maximum lengths and are returned in a format that the Hugging Face Trainer can directly use for fine-tuning.

In [27]:
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
ft_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Train/validation split
alice_split = filtered.train_test_split(test_size=0.2, seed=42)
train_ds = alice_split["train"]
eval_ds = alice_split["test"]

max_input_length = 256
max_target_length = 64

def preprocess_batch(batch):
    inputs = [
        f"question: {q} context: {c}"
        for q, c in zip(batch["question"], batch["context"])
    ]
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["answer"],
            max_length=max_target_length,
            truncation=True,
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

The next code block prepares the dataset for training and launches the fine-tuning process using Hugging Face’s Trainer API. The map calls apply the preprocess_batch function to every example in the training and validation splits, producing tokenized versions of each dataset and removing the original text columns. A DataCollatorForSeq2Seq is created to dynamically pad inputs and labels to the correct lengths during batching. The TrainingArguments block specifies key hyperparameters such as learning rate, batch size, number of epochs, weight decay & logging frequency. Finally, a Trainer object is instantiated with the model, training arguments, tokenized datasets, data collator, and tokenizer. Calling trainer.train() begins supervised fine-tuning of the FLAN-T5 model on the Alice QA pairs.

In [29]:
tokenized_train = train_ds.map(preprocess_batch, batched=True, remove_columns=train_ds.column_names)
tokenized_eval = eval_ds.map(preprocess_batch, batched=True, remove_columns=eval_ds.column_names)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=ft_model)

training_args = TrainingArguments(
    output_dir="./flan_t5_alice",
    evaluation_strategy="epoch",
    learning_rate=5e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_total_limit=1,
)

trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,1.621645
2,No log,1.628938
3,No log,1.657578


TrainOutput(global_step=39, training_loss=1.249781486315605, metrics={'train_runtime': 10.9351, 'train_samples_per_second': 26.886, 'train_steps_per_second': 3.566, 'total_flos': 9981448740864.0, 'train_loss': 1.249781486315605, 'epoch': 3.0})

#### Hyperparameter set #1 performance
The training hyperparameters were chosen to balance learning signal with speed and overfitting risk on a very small Alice in Wonderland QA dataset. I used a relatively modest learning rate of 5e-4, which is common for fine-tuning encoder–decoder models and allows the model to adapt without aggressively overwriting its pretrained weights. A batch size of 8 fits comfortably on typical notebook hardware while still giving stable gradient estimates. The model is trained for 3 epochs, which is a conservative choice given the small dataset size. I enabled evaluation at the end of each epoch (evaluation_strategy="epoch") to monitor validation loss over time, added weight decay of 0.01 to provide a bit of regularization, and limited checkpoints to a single copy (save_total_limit=1) to keep disk usage low. Overall, the configuration is designed as a reasonable “first pass” that runs quickly and establishes a baseline for more aggressive tuning.

#### Results
The training output shows that the model completed 3 epochs in only ~20 seconds, confirming that this setup is computationally light and can easily support more extensive experimentation. The final training loss is about 1.25, with validation losses hovering around 1.62-1.66 across epochs. The fact that validation loss does not dramatically improve or deteriorate suggests that the model is learning something useful from the Alice QA pairs but may be approaching a plateau with this particular configuration and dataset size. Because training is so fast, it would be reasonable to explore more in-depth settings such as increasing the number of epochs (e.g., 5–10), trying a slightly smaller learning rate, or adding early stopping based on validation loss to see whether performance on held-out Alice questions can be pushed further without overfitting.

#### Hyperparameter set #2

In [33]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=ft_model)

training_args = TrainingArguments(
    output_dir="./flan_t5_alice_deeper",
    evaluation_strategy="epoch",
    save_strategy="epoch",                 # save at each epoch so early stopping can pick best
    learning_rate=3e-4,                    # slightly smaller LR for more stable fine-tuning
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=8,                    # allow more passes
    weight_decay=0.01,                     # regularization
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,           # reload best checkpoint after training
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    tokenizer=tokenizer,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2)  # stop if eval loss doesn’t improve for 2 epochs
    ],
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,1.789680
2,No log,1.846641
3,No log,2.013547


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=39, training_loss=0.45043231279422075, metrics={'train_runtime': 7.5456, 'train_samples_per_second': 103.902, 'train_steps_per_second': 13.783, 'total_flos': 9981448740864.0, 'train_loss': 0.45043231279422075, 'epoch': 3.0})

The new training run shows that with the deeper schedule (up to 8 epochs allowed, smaller learning rate, early stopping) the model actually overfit faster rather than improving generalization. You can see this in the validation loss: it starts around 1.79 at epoch 1, then increases to ~1.85 and ~2.02 by epoch 3, even though the training loss keeps dropping to 0.45. That low training loss but worsening validation loss means the model is fitting the training Alice QA pairs more tightly while getting worse on the held-out eval set. Because of load_best_model_at_end=True and the early-stopping callback, the Trainer will reload the checkpoint from the epoch with the lowest validation loss (epoch 1 in this case), so you still end up with the best of these runs even though later epochs hurt performance.

The takeaway here is that simply training longer and harder on a small dataset doesn’t guarantee better results, and in this case the earlier, lighter fine-tuning configuration (or epoch 1 of this run) likely gives better validation performance. That’s a good motivation for combining modest fine-tuning with smarter prompting or retrieval-augmented QA, rather than just cranking up epochs.

#### Final Tuned Model
Since the first hyperparameter set resulted in lower validation loss, let's retrain the model using the previous hyperparameters:

In [36]:
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
ft_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=ft_model)

training_args = TrainingArguments(
    output_dir="./flan_t5_alice_final",
    evaluation_strategy="epoch",
    save_strategy="epoch",          # save each epoch
    learning_rate=5e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_total_limit=1,             # only keep best/last
    load_best_model_at_end=True,    # reload best val-loss checkpoint
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

trainer.train()

# Build a pipeline for the optimized, fine-tuned model
alice_qa_pipe = pipeline(
    "text2text-generation",
    model=ft_model,
    tokenizer=tokenizer,
)

Epoch,Training Loss,Validation Loss
1,No log,1.621646
2,No log,1.628925
3,No log,1.657275


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


### Prompt Engineering

To better understand how different optimization techniques affect question-answering performance, I'll evaluate three progressively more informed versions of the model on the final Alice in Wonderland questions. First, I test the fine-tuned model alone to establish a baseline after the supervised training above. Next, I introduce prompt-engineering through few-shot examples, allowing the tuned model to see several in-domain QA pairs before generating an answer. Finally, I test a retrieval-augmented approach, where the tuned model is provided with a relevant context passage from the Alice dataset based on similarity to the target question. Comparing these three strategies highlights how additional task guidance and contextual support can improve accuracy beyond fine-tuning alone.

In [39]:
few_shot_examples = filtered.select(range(3))  

def answer_with_few_shot(question, num_examples=3, max_new_tokens=40):
    prefix = "You are answering questions about Alice's Adventures in Wonderland.\n\n"
    for ex in filtered.select(range(num_examples)):
        prefix += f"Q: {ex['question']}\nA: {ex['answer']}\n\n"
    prompt = prefix + f"Q: {question}\nA:"
    out = alice_qa_pipe(prompt, max_new_tokens=max_new_tokens)
    return out[0]["generated_text"]

This code above implements a few-shot prompting strategy, where a small number of sample QA pairs from the Alice dataset are prepended to the model’s input before asking the target question. By supplying 2-3 example questions along with their correct answers, the prompt establishes an in-domain pattern and demonstrates the type of reasoning the model should perform. The function then appends the new question and lets the fine-tuned model generate the answer. This approach does not modify the model weights, instead leveraging the model’s instruction-following behavior to guide it toward more accurate responses.

In [41]:
def retrieve_best_context(query, dataset):
    best_score = -1.0
    best_context = ""
    for ex in dataset:
        score = SequenceMatcher(None, query.lower(), ex["question"].lower()).ratio()
        if score > best_score:
            best_score = score
            best_context = ex["context"]
    return best_context

def answer_with_retrieval(question, max_new_tokens=60):
    context = retrieve_best_context(question, filtered)
    prompt = (
        "Use the context to answer the question.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )
    out = alice_qa_pipe(prompt, max_new_tokens=max_new_tokens)
    return out[0]["generated_text"]

This final code block adds a retrieval layer that automatically selects the most similar question from the Alice dataset and extracts its associated context passage. That context is then included directly in the prompt as supporting information. By providing the model with relevant text from the source material, the retrieval-augmented approach reduces reliance on the model’s internal knowledge and helps prevent hallucinations. The model then answers the question using both its fine-tuned parameters and the retrieved context.

# __6. Asking the Same Questions Again (After Fine-Tuning or Better Context)__<br>
Re-running the same questions from before, post-Training and Prompt Engineering.

To evaluate how each optimization strategy affected performance, I re-ran the same six Alice in Wonderland questions using three different versions of the model: the fine-tuned model alone, the fine-tuned model with few-shot prompt engineering, and the fine-tuned model with retrieval-augmented context. For each question, I also provide the correct (gold) answer (that I looked up online) to facilitate direct comparison.

In [45]:
# Questions and their correct answers
qa_gold = {
    "What is the White Rabbit's catchphrase?": 
        "Oh dear! Oh dear! I shall be late!",
    
    "Why does Alice shrink?":
        "She shrinks after drinking from a bottle labeled 'Drink Me.'",
    
    "What is the Queen's full name?":
        "The Queen of Hearts.", 
    
    "What game does Alice play?":
        "Croquet.",
    
    "Who wrote Alice's Adventures in Wonderland?":
        "Lewis Carroll.",
    
    "Who played the Mad Hatter in Tim Burton's film version of Alice's Adventures in Wonderland?":
        "Johnny Depp."
}

novel_questions = list(qa_gold.keys())

def answer_finetuned_only(question):
    prompt = f"question: {question}"
    out = alice_qa_pipe(prompt, max_new_tokens=40)
    return out[0]["generated_text"]

for q in novel_questions:
    correct = qa_gold[q]
    ft_ans   = answer_finetuned_only(q)
    few_ans  = answer_with_few_shot(q)
    retr_ans = answer_with_retrieval(q)

    print(f"Q: {q}")
    print(f"  Correct answer:       {correct}")
    print(f"  Fine-tuned only:      {ft_ans}")
    print(f"  + Few-shot examples:  {few_ans}")
    print(f"  + Retrieval context:  {retr_ans}")
    print("-" * 100)

Q: What is the White Rabbit's catchphrase?
  Correct answer:       Oh dear! Oh dear! I shall be late!
  Fine-tuned only:      The White Rabbit
  + Few-shot examples:  The White Rabbit's catchphrase is 'The White Rabbit's catchphrase'.
  + Retrieval context:  The White Rabbit lost a fan and a pair of white kid gloves.
----------------------------------------------------------------------------------------------------
Q: Why does Alice shrink?
  Correct answer:       She shrinks after drinking from a bottle labeled 'Drink Me.'
  Fine-tuned only:      Because she is a sexy, sexy, sexy, sexy, sexy, sexy, sexy,
  + Few-shot examples:  Because she is shrinking.
  + Retrieval context:  Because she is a white Rabbit.
----------------------------------------------------------------------------------------------------
Q: What is the Queen's full name?
  Correct answer:       The Queen of Hearts.
  Fine-tuned only:      Queen Elizabeth II
  + Few-shot examples:  Queen Victoria
  + Retrieval conte

Across all six questions, none of the three methods, fine-tuning alone, prompt engineering, or retrieval augmentation, produced accurate answers, which may be expected for a very small model like FLAN-T5-Small. Because the questions require either detailed story-specific memory (“What game does Alice play?”) or external factual knowledge (“Who played the Mad Hatter in Tim Burton’s film?”), the model simply does not have the capacity, knowledge depth, or pretrained intellect needed to answer them correctly. Fine-tuning helped the model become more “on-theme” (e.g., shifting answers toward characters from the story rather than random city names), but it did not meaningfully improve factual correctness because the Alice dataset contains limited coverage, with short, noisy contexts that don’t explicitly supply the needed answers.

Prompt engineering showed slight behavioral shifts as answers became more structured and story-related, but it did not fix the underlying knowledge gap. Retrieval augmentation often made answers worse, because the similarity-based retrieval returned context passages that were irrelevant to the actual question (e.g., a random paragraph about the White Rabbit losing his gloves), causing the model to anchor on misleading text. Since the retrieval system was naive (string similarity only), the context rarely aligned with the question’s actual answer, and the model predictably hallucinated. I think this dataset is simply too small and too narrow to properly apply an effective augmented retrieval strategy.

Overall, performance did not improve in a measurable accuracy sense, but the experiment highlights that small models cannot acquire deep world knowledge or precise recall from fine-tuning on limited datasets. In practice, solving these questions would require either (1) a much larger model, (2) a rich retrieval system tied to the full text of the novel, or (3) far more training examples spanning every detail of the story. What these results do show is how each method influences the model’s behavior, even when correctness does not improve.

# References

- Natural Language Processing with Python. NLTK Project, https://www.nltk.org/book/.
- Hugging Face. Hugging Face. https://huggingface.co.
- Hugging Face. google/flan-t5-small. https://huggingface.co/google/flan-t5-small.
- Hugging Face Datasets. deepmind/narrativeqa. https://huggingface.co/datasets/deepmind/narrativeqa.
- Project Gutenberg. https://www.gutenberg.org/